In [1]:
import os

In [2]:
%pwd

'c:\\Users\\lenov\\OneDrive\\Documents\\projects\\Kidney-disease-classification-Deep-learning-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\lenov\\OneDrive\\Documents\\projects\\Kidney-disease-classification-Deep-learning-project'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/rohit-patil-2507/Kidney-disease-classification-Deep-learning-project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="rohit-patil-2507"
os.environ["MLFLOW_TRACKING_PASSWORD"]="6fa804be4255ec76efdfbf2a6afa6e530097b7a2"

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/rohit-patil-2507/Kidney-disease-classification-Deep-learning-project.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse
from cnnClassifier  import logger

In [12]:


class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        tracking_uri = os.getenv("MLFLOW_TRACKING_URI", self.config.mlflow_uri)
        mlflow.set_tracking_uri(tracking_uri)

        experiment_name = "Kidney-Disease-Classification"
        
        # Check if experiment exists and is not deleted
        exp = mlflow.get_experiment_by_name(experiment_name)
        if exp is not None and exp.lifecycle_stage == "deleted":
            # Restore the deleted experiment using MlflowClient
            client = MlflowClient()
            client.restore_experiment(exp.experiment_id)
        elif exp is None:
            # Create new experiment if it doesn't exist
            mlflow.create_experiment(experiment_name)
        
        mlflow.set_experiment(experiment_name)
        
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )

            log_model = os.getenv(
                "MLFLOW_LOG_MODEL",
                "true" if tracking_url_type_store == "file" else "false"
            ).lower()

            if log_model in ("0", "false", "no"):
                logger.info(
                    "Skipping MLflow model artifact logging. Set MLFLOW_LOG_MODEL=true to upload/register the model."
                )
                return

            try:
                # Register model only if using remote tracking server
                if tracking_url_type_store != "file":
                    mlflow.keras.log_model(
                        self.model,
                        "model",
                        registered_model_name="VGG16Model"
                    )
                else:
                    mlflow.keras.log_model(self.model, "model")
            except Exception as e:
                logger.warning(f"MLflow params and metrics were logged, but model artifact upload failed: {e}")


In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    # evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-05-09 15:12:37,464: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-09 15:12:37,475: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-09 15:12:37,477: INFO: common: created directory at: artifacts]
Found 600 images belonging to 2 classes.
20/20 [==============================] - 12s 131ms/step - loss: 0.1165 - accuracy: 0.9583 - precision: 0.9583 - recall: 0.9583 - auc: 0.9959
[2026-05-09 15:12:49,740: INFO: common: json file saved at: scores.json]


In [14]:
# from mlflow import MlflowClient
# client = MlflowClient()
# client.set_registered_model_alias("VGG16Model", "Staging", 1)  # Assigns alias 'production' to version 1


In [ ]:
from mlflow import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(
    name="VGG16Model",
    version="7",
    stage="Production",
    archive_existing_versions=False  # Optional: archives previous versions in this stage
)


C:\Users\lenov\AppData\Local\Temp\ipykernel_22640\112411515.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778092815531, current_stage='Production', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1778319771419, metrics=None, model_id=None, name='VGG16Model', params=None, run_id='cd3be21ad36042e08f5607342ebb8a31', run_link='', source='models:/m-5e76192b047b45518f9ca18c764a504b', status='READY', status_message=None, tags={}, user_id='', version='7', workspace='default'>

: 